## Logistic Regression 

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from arson.configs.settings import project_dir

In [2]:
data_dir = project_dir.RAW_DIR

In [3]:
dataset_path_train = data_dir/'application_train.csv'
dataset_path_test = data_dir/'application_test.csv'

df_train = pd.read_csv(dataset_path_train)
df_test = pd.read_csv(dataset_path_test)



In [4]:
df_train.sample(4)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
143564,266475,0,Cash loans,F,N,Y,0,99000.0,157500.0,7065.0,...,0,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0
43529,150393,0,Cash loans,F,N,Y,0,202500.0,225000.0,23755.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
146791,270193,1,Cash loans,M,Y,Y,0,225000.0,783000.0,21532.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
206559,339418,0,Cash loans,F,N,Y,1,180000.0,450000.0,27324.0,...,0,0,0,0,0.0,0.0,0.0,1.0,0.0,1.0


In [5]:
## dropping the unwanted columns
df_col_path = data_dir/'col_description.csv'
df_col = pd.read_csv(df_col_path,encoding='latin1',index_col=0)
df_col = df_col[df_col['Table']=='application_{train|test}.csv']
df_col.sample(3)


,Table,Row,Description,Special
75,application_{train|test}.csv,APARTMENTS_MEDI,Normalized information about building where th...,normalized
90,application_{train|test}.csv,HOUSETYPE_MODE,Normalized information about building where th...,normalized
76,application_{train|test}.csv,BASEMENTAREA_MEDI,Normalized information about building where th...,normalized


In [6]:
avg_columns = df_train.columns.str.contains('AVG')
avg_columns_name = df_train.columns[avg_columns]
avg_columns_name

Index(['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG',
       'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG',
       'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG',
       'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG',
       'NONLIVINGAREA_AVG'],
      dtype='str')

In [7]:
# median columns
med_cols = df_train.columns.str.contains('MEDI')
med_cols_name = df_train.columns[med_cols]
med_cols_name

Index(['APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI',
       'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI',
       'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI',
       'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI',
       'NONLIVINGAREA_MEDI'],
      dtype='str')

In [8]:
## mode columns
mode_cols = df_train.columns.str.contains('MOD')
med_cols_name = df_train.columns[mode_cols]
med_cols_name

Index(['APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE',
       'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE',
       'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE',
       'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE',
       'NONLIVINGAREA_MODE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE',
       'TOTALAREA_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE'],
      dtype='str')

In [9]:
## flag columns
flag_cols = df_train.columns.str.contains('DOCUMENT')
flag_cols_name = df_train.columns[flag_cols]
flag_cols_name

Index(['FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4',
       'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7',
       'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10',
       'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13',
       'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16',
       'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19',
       'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21'],
      dtype='str')

In [10]:
## Bureau columns
bureau_col = df_col[df_col['Row'].str.contains('BUREAU')]
bureau_col_names = bureau_col['Row']
bureau_col_names

119    AMT_REQ_CREDIT_BUREAU_HOUR
120     AMT_REQ_CREDIT_BUREAU_DAY
121    AMT_REQ_CREDIT_BUREAU_WEEK
122     AMT_REQ_CREDIT_BUREAU_MON
123     AMT_REQ_CREDIT_BUREAU_QRT
124    AMT_REQ_CREDIT_BUREAU_YEAR
Name: Row, dtype: str

In [11]:
df_train = df_train.drop(flag_cols_name,axis=1)
df_test = df_test.drop(flag_cols_name,axis=1)

In [12]:
df_train

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,2.0,2.0,2.0,-1134.0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,1.0,0.0,-828.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,0.0,-815.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,2.0,0.0,-617.0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,0.0,-1106.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0.0,0.0,0.0,-273.0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0.0,6.0,0.0,-1909.0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0.0,0.0,0.0,-322.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
noise_col = df_col[df_col['Row'].str.contains('AVG|MODE|MEDI')]
noise_col

,Table,Row,Description,Special
47,application_{train|test}.csv,APARTMENTS_AVG,Normalized information about building where th...,normalized
48,application_{train|test}.csv,BASEMENTAREA_AVG,Normalized information about building where th...,normalized
49,application_{train|test}.csv,YEARS_BEGINEXPLUATATION_AVG,Normalized information about building where th...,normalized
50,application_{train|test}.csv,YEARS_BUILD_AVG,Normalized information about building where th...,normalized
51,application_{train|test}.csv,COMMONAREA_AVG,Normalized information about building where th...,normalized
52,application_{train|test}.csv,ELEVATORS_AVG,Normalized information about building where th...,normalized
53,application_{train|test}.csv,ENTRANCES_AVG,Normalized information about building where th...,normalized
54,application_{train|test}.csv,FLOORSMAX_AVG,Normalized information about building where th...,normalized
55,application_{train|test}.csv,FLOORSMIN_AVG,Normalized information about building where th...,normalized
56,application_{train|test}.csv,LANDAREA_AVG,Normalized information about building where th...,normalized


In [14]:
noise_col_names = noise_col['Row']
noise_col_names

47                  APARTMENTS_AVG
48                BASEMENTAREA_AVG
49     YEARS_BEGINEXPLUATATION_AVG
50                 YEARS_BUILD_AVG
51                  COMMONAREA_AVG
52                   ELEVATORS_AVG
53                   ENTRANCES_AVG
54                   FLOORSMAX_AVG
55                   FLOORSMIN_AVG
56                    LANDAREA_AVG
57            LIVINGAPARTMENTS_AVG
58                  LIVINGAREA_AVG
59         NONLIVINGAPARTMENTS_AVG
60               NONLIVINGAREA_AVG
61                 APARTMENTS_MODE
62               BASEMENTAREA_MODE
63    YEARS_BEGINEXPLUATATION_MODE
64                YEARS_BUILD_MODE
65                 COMMONAREA_MODE
66                  ELEVATORS_MODE
67                  ENTRANCES_MODE
68                  FLOORSMAX_MODE
69                  FLOORSMIN_MODE
70                   LANDAREA_MODE
71           LIVINGAPARTMENTS_MODE
72                 LIVINGAREA_MODE
73        NONLIVINGAPARTMENTS_MODE
74              NONLIVINGAREA_MODE
75                 A

In [15]:
df_train.drop(noise_col_names,axis=1,inplace=True,errors='ignore')
df_test = df_test.drop(noise_col_names,axis=1)

In [16]:
print(df_train.shape)
print(df_test.shape)

(307511, 55)
(48744, 54)


In [17]:
df_train.sample()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
212368,346102,0,Revolving loans,M,N,Y,1,225000.0,270000.0,13500.0,...,0.0,0.0,0.0,-1515.0,0.0,0.0,0.0,1.0,1.0,2.0


In [18]:
### only categorical columns
cat_col = df_train.select_dtypes('str')
cat_col.shape

(307511, 12)

In [19]:
### Numercial columns
num_col = df_train.select_dtypes(['int64','float64'])
num_col.shape

(307511, 43)

In [20]:
cat_col_names = cat_col.columns.to_list()
cat_col_names

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE']

In [21]:
num_col_names = num_col.columns.to_list()
num_col_names

['SK_ID_CURR',
 'TARGET',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'OBS_30_CNT_SOCIAL_CIRCLE',
 'DEF_30_CNT_SOCIAL_CIRCLE',
 'OBS_60_CNT_SOCIAL_CIRCLE',
 'DEF_60_CNT_SOCIAL_CIRCLE',
 'DAYS_LAST_PHONE_CHANGE',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_YEAR']

In [22]:
df_train[num_col_names]

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,0,202500.0,406597.5,24700.5,351000.0,0.018801,-9461,-637,...,2.0,2.0,2.0,-1134.0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-16765,-1188,...,0.0,1.0,0.0,-828.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,0,67500.0,135000.0,6750.0,135000.0,0.010032,-19046,-225,...,0.0,0.0,0.0,-815.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,0,135000.0,312682.5,29686.5,297000.0,0.008019,-19005,-3039,...,0.0,2.0,0.0,-617.0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,0,121500.0,513000.0,21865.5,513000.0,0.028663,-19932,-3038,...,0.0,0.0,0.0,-1106.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,0,157500.0,254700.0,27558.0,225000.0,0.032561,-9327,-236,...,0.0,0.0,0.0,-273.0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,0,72000.0,269550.0,12001.5,225000.0,0.025164,-20775,365243,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,0,153000.0,677664.0,29979.0,585000.0,0.005002,-14966,-7921,...,0.0,6.0,0.0,-1909.0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,0,171000.0,370107.0,20205.0,319500.0,0.005313,-11961,-4786,...,0.0,0.0,0.0,-322.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
df_train.drop('SK_ID_CURR',axis=1,inplace=True)

In [24]:
missing = df_train.isnull().sum().sort_values(ascending=False).to_frame()
missing['percentage'] = round(missing/len(df_train) * 100,2)
missing

,0,percentage
OWN_CAR_AGE,202929,65.99
EXT_SOURCE_1,173378,56.38
OCCUPATION_TYPE,96391,31.35
EXT_SOURCE_3,60965,19.83
AMT_REQ_CREDIT_BUREAU_HOUR,41519,13.50
AMT_REQ_CREDIT_BUREAU_QRT,41519,13.50
AMT_REQ_CREDIT_BUREAU_WEEK,41519,13.50
AMT_REQ_CREDIT_BUREAU_MON,41519,13.50
AMT_REQ_CREDIT_BUREAU_YEAR,41519,13.50
AMT_REQ_CREDIT_BUREAU_DAY,41519,13.50


In [25]:
df_col[df_col['Row'].str.contains('AMT')]

,Table,Row,Description,Special
10,application_{train|test}.csv,AMT_INCOME_TOTAL,Income of the client,NaN
11,application_{train|test}.csv,AMT_CREDIT,Credit amount of the loan,NaN
12,application_{train|test}.csv,AMT_ANNUITY,Loan annuity,NaN
13,application_{train|test}.csv,AMT_GOODS_PRICE,For consumer loans it is the price of the good...,NaN
119,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_HOUR,Number of enquiries to Credit Bureau about the...,NaN
120,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_DAY,Number of enquiries to Credit Bureau about the...,NaN
121,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_WEEK,Number of enquiries to Credit Bureau about the...,NaN
122,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_MON,Number of enquiries to Credit Bureau about the...,NaN
123,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_QRT,Number of enquiries to Credit Bureau about the...,NaN
124,application_{train|test}.csv,AMT_REQ_CREDIT_BUREAU_YEAR,Number of enquiries to Credit Bureau about the...,NaN


In [26]:
bureau_col = df_col[df_col['Row'].str.contains('BUREAU')]['Row']
bureau_col

119    AMT_REQ_CREDIT_BUREAU_HOUR
120     AMT_REQ_CREDIT_BUREAU_DAY
121    AMT_REQ_CREDIT_BUREAU_WEEK
122     AMT_REQ_CREDIT_BUREAU_MON
123     AMT_REQ_CREDIT_BUREAU_QRT
124    AMT_REQ_CREDIT_BUREAU_YEAR
Name: Row, dtype: str

In [27]:
df_train.groupby(bureau_col)['TARGET'].agg({
    'count','mean'
})

,mean,count
Row,,
AMT_REQ_CREDIT_BUREAU_DAY,0.0,1
AMT_REQ_CREDIT_BUREAU_HOUR,0.0,1
AMT_REQ_CREDIT_BUREAU_MON,0.0,1
AMT_REQ_CREDIT_BUREAU_QRT,0.0,1
AMT_REQ_CREDIT_BUREAU_WEEK,0.0,1
AMT_REQ_CREDIT_BUREAU_YEAR,0.0,1


In [28]:
missing

,0,percentage
OWN_CAR_AGE,202929,65.99
EXT_SOURCE_1,173378,56.38
OCCUPATION_TYPE,96391,31.35
EXT_SOURCE_3,60965,19.83
AMT_REQ_CREDIT_BUREAU_HOUR,41519,13.50
AMT_REQ_CREDIT_BUREAU_QRT,41519,13.50
AMT_REQ_CREDIT_BUREAU_WEEK,41519,13.50
AMT_REQ_CREDIT_BUREAU_MON,41519,13.50
AMT_REQ_CREDIT_BUREAU_YEAR,41519,13.50
AMT_REQ_CREDIT_BUREAU_DAY,41519,13.50


In [29]:
p_dir = project_dir
processed_dir = p_dir.PROCESSED_DIR

df_col.to_csv(processed_dir/'df_col_desc.csv',index=False)

# processed_dir

In [30]:
missing

,0,percentage
OWN_CAR_AGE,202929,65.99
EXT_SOURCE_1,173378,56.38
OCCUPATION_TYPE,96391,31.35
EXT_SOURCE_3,60965,19.83
AMT_REQ_CREDIT_BUREAU_HOUR,41519,13.50
AMT_REQ_CREDIT_BUREAU_QRT,41519,13.50
AMT_REQ_CREDIT_BUREAU_WEEK,41519,13.50
AMT_REQ_CREDIT_BUREAU_MON,41519,13.50
AMT_REQ_CREDIT_BUREAU_YEAR,41519,13.50
AMT_REQ_CREDIT_BUREAU_DAY,41519,13.50


In [31]:
df_train.columns[df_train.columns.str.contains('CAR')]

Index(['FLAG_OWN_CAR', 'OWN_CAR_AGE'], dtype='str')

In [32]:
df_train.groupby('OWN_CAR_AGE')['TARGET'].agg({
   'mean','count'
})

,mean,count
OWN_CAR_AGE,,
0.0,0.068885,2134
1.0,0.063636,5280
2.0,0.059979,5852
3.0,0.049922,6370
4.0,0.053986,5557
...,...,...
63.0,0.000000,2
64.0,0.085960,2443
65.0,0.077441,891


In [33]:
df_train[['OWN_CAR_AGE','FLAG_OWN_CAR']].sample(4)
# df_train.groupby('FLAG_OWN_CAR')['OWN_CAR_AGE']
## we are going to drop own car age col, as it doesnt have any relation withh our target variable
df_train.drop('OWN_CAR_AGE',axis=1,inplace=True)
df_test = df_test.drop('OWN_CAR_AGE',axis=1)

In [34]:
df_train.groupby('FLAG_OWN_CAR')['TARGET'].count()

FLAG_OWN_CAR
N    202924
Y    104587
Name: TARGET, dtype: int64

In [35]:
missing

,0,percentage
OWN_CAR_AGE,202929,65.99
EXT_SOURCE_1,173378,56.38
OCCUPATION_TYPE,96391,31.35
EXT_SOURCE_3,60965,19.83
AMT_REQ_CREDIT_BUREAU_HOUR,41519,13.50
AMT_REQ_CREDIT_BUREAU_QRT,41519,13.50
AMT_REQ_CREDIT_BUREAU_WEEK,41519,13.50
AMT_REQ_CREDIT_BUREAU_MON,41519,13.50
AMT_REQ_CREDIT_BUREAU_YEAR,41519,13.50
AMT_REQ_CREDIT_BUREAU_DAY,41519,13.50


### missing value and customer does not own a car, so it is actually not a missingness in our data, this is our insights

In [36]:
df_train['OCCUPATION_TYPE'].value_counts()

OCCUPATION_TYPE
Laborers                 55186
Sales staff              32102
Core staff               27570
Managers                 21371
Drivers                  18603
High skill tech staff    11380
Accountants               9813
Medicine staff            8537
Security staff            6721
Cooking staff             5946
Cleaning staff            4653
Private service staff     2652
Low-skill Laborers        2093
Waiters/barmen staff      1348
Secretaries               1305
Realty agents              751
HR staff                   563
IT staff                   526
Name: count, dtype: int64

In [37]:
df_train.groupby('OCCUPATION_TYPE')['TARGET'].agg({
    'count','mean'
})

,mean,count
OCCUPATION_TYPE,,
Accountants,0.048303,9813
Cleaning staff,0.096067,4653
Cooking staff,0.104440,5946
Core staff,0.063040,27570
Drivers,0.113261,18603
HR staff,0.063943,563
High skill tech staff,0.061599,11380
IT staff,0.064639,526
Laborers,0.105788,55186


In [38]:
df_train['OCCUPATION_TYPE'].isna().mean()

np.float64(0.31345545362604915)

In [39]:
df_train.groupby(
    df_train['OCCUPATION_TYPE'].isna()
)['TARGET'].mean()


OCCUPATION_TYPE
False    0.087851
True     0.065131
Name: TARGET, dtype: float64

In [40]:
df_train['OCCUPATION_TYPE'] = df_train['OCCUPATION_TYPE'].fillna('Unknown')
df_test['OCCUPATION_TYEP'] = df_test['OCCUPATION_TYPE'].fillna('Unknown')



In [41]:
df_train['OCCUPATION_TYPE'].isna().mean()

np.float64(0.0)

In [42]:
## handling the missing value in the flag own car column
car_mode = df_train['FLAG_OWN_CAR'].mode()[0]
df_train['FLAG_OWN_CAR'] = df_train['FLAG_OWN_CAR'].fillna(car_mode)
df_test['FLAG_OWN_CAR'] = df_test['FLAG_OWN_CAR'].fillna(car_mode)

In [43]:
df_train['FLAG_OWN_CAR'] = df_train['FLAG_OWN_CAR'].map({
    'Y': 1,
    'N' : 0
})
df_test['FLAG_OWN_CAR'] = df_test['FLAG_OWN_CAR'].map({
    'Y': 1,
    'N' : 0
})

In [44]:
df_train.isnull().sum().sort_values(ascending=False)

EXT_SOURCE_1                   173378
EXT_SOURCE_3                    60965
AMT_REQ_CREDIT_BUREAU_DAY       41519
AMT_REQ_CREDIT_BUREAU_HOUR      41519
AMT_REQ_CREDIT_BUREAU_QRT       41519
AMT_REQ_CREDIT_BUREAU_WEEK      41519
AMT_REQ_CREDIT_BUREAU_MON       41519
AMT_REQ_CREDIT_BUREAU_YEAR      41519
NAME_TYPE_SUITE                  1292
DEF_30_CNT_SOCIAL_CIRCLE         1021
OBS_30_CNT_SOCIAL_CIRCLE         1021
DEF_60_CNT_SOCIAL_CIRCLE         1021
OBS_60_CNT_SOCIAL_CIRCLE         1021
EXT_SOURCE_2                      660
AMT_GOODS_PRICE                   278
AMT_ANNUITY                        12
CNT_FAM_MEMBERS                     2
DAYS_LAST_PHONE_CHANGE              1
FLAG_OWN_REALTY                     0
NAME_CONTRACT_TYPE                  0
TARGET                              0
FLAG_MOBIL                          0
DAYS_ID_PUBLISH                     0
DAYS_REGISTRATION                   0
DAYS_EMPLOYED                       0
DAYS_BIRTH                          0
REGION_POPUL

In [45]:
df_train.dropna(subset=['AMT_GOODS_PRICE'],inplace=True)
df_test.dropna(subset=['AMT_GOODS_PRICE'],inplace=True)

In [46]:
df_train.isnull().sum().sort_values(ascending=False)

EXT_SOURCE_1                   173234
EXT_SOURCE_3                    60897
AMT_REQ_CREDIT_BUREAU_DAY       41473
AMT_REQ_CREDIT_BUREAU_HOUR      41473
AMT_REQ_CREDIT_BUREAU_QRT       41473
AMT_REQ_CREDIT_BUREAU_WEEK      41473
AMT_REQ_CREDIT_BUREAU_MON       41473
AMT_REQ_CREDIT_BUREAU_YEAR      41473
OBS_60_CNT_SOCIAL_CIRCLE         1021
DEF_30_CNT_SOCIAL_CIRCLE         1021
OBS_30_CNT_SOCIAL_CIRCLE         1021
DEF_60_CNT_SOCIAL_CIRCLE         1021
NAME_TYPE_SUITE                  1014
EXT_SOURCE_2                      659
AMT_ANNUITY                        12
DAYS_LAST_PHONE_CHANGE              1
FLAG_OWN_CAR                        0
FLAG_OWN_REALTY                     0
CODE_GENDER                         0
NAME_CONTRACT_TYPE                  0
TARGET                              0
FLAG_MOBIL                          0
DAYS_ID_PUBLISH                     0
DAYS_REGISTRATION                   0
DAYS_EMPLOYED                       0
DAYS_BIRTH                          0
REGION_POPUL

In [47]:
cat_col = df_train.select_dtypes('str')
num_col = df_train.select_dtypes('int64','float64')
num_col_names = num_col.columns
cat_col_names = cat_col.columns

In [48]:
df_train.shape

(307233, 53)

### Building pipeline

### columns we are gonna use
-> CODE_GENDER , FLAG_OWN_CAR, CNT_CHILDREN, OCCUPATIOn_TYPE, REGION_RATING_CLIENT, AMT_GOODS_PRICE, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY

In [49]:
features = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'CNT_CHILDREN',
    'OCCUPATION_TYPE',
    'REGION_RATING_CLIENT',
    'AMT_GOODS_PRICE',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY'
]

X = df_train[features]
y = df_train['TARGET']

In [50]:
categorical_features = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'OCCUPATION_TYPE'
]

numerical_features = [
    'CNT_CHILDREN',
    'REGION_RATING_CLIENT',
    'AMT_GOODS_PRICE',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY'
]

In [51]:
X.info()

<class 'pandas.DataFrame'>
Index: 307233 entries, 0 to 307510
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   CODE_GENDER           307233 non-null  str    
 1   FLAG_OWN_CAR          307233 non-null  int64  
 2   CNT_CHILDREN          307233 non-null  int64  
 3   OCCUPATION_TYPE       307233 non-null  str    
 4   REGION_RATING_CLIENT  307233 non-null  int64  
 5   AMT_GOODS_PRICE       307233 non-null  float64
 6   AMT_INCOME_TOTAL      307233 non-null  float64
 7   AMT_CREDIT            307233 non-null  float64
 8   AMT_ANNUITY           307221 non-null  float64
dtypes: float64(4), int64(3), str(2)
memory usage: 23.4 MB


In [52]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])



In [53]:
cat_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

In [54]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('num',num_pipeline,numerical_features),
    ('cat',cat_pipeline,categorical_features)
])

In [55]:
from sklearn.model_selection import train_test_split
X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.20, random_state=42, stratify=y)

In [57]:
preprocessor.fit(X_train, y_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

## building logistic regression pipeline

In [59]:
from sklearn.linear_model import LogisticRegression

model_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',LogisticRegression(max_iter=1000))
])

In [60]:
model_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the

In [61]:
model_pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['CODE_GENDER','FLAG_OWN_CAR','CNT_CHILDREN',...,'AMT_INCOME_TOTAL', 'AMT_CREDIT','AMT_ANNUITY']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainde

In [66]:
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:,1]

In [69]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score


print(f"Accuracy: {accuracy_score(y_test,y_pred)}")

print(f"ROC AUC Score: {roc_auc_score(y_test,y_prob)}")

Accuracy: 0.9192637557569938
ROC AUC Score: 0.6268198859006635
